In [ ]:
import tensorflow as tf
import numpy as np
from tensorflow import keras
from google.colab import drive
import shutil
import cv2
import pandas as pd
import matplotlib.pyplot as plt
import os
from tensorflow.keras.applications.efficientnet_v2 import preprocess_input
import seaborn as sns

In [ ]:
drive.mount('/content/drive/', force_remount=False)
shutil.copy("/content/drive/MyDrive/kaggle.zip","/content/")
!unzip kaggle.zip

In [ ]:
def ArreglarLasRutas(df):
    rutas_nuevas = []
    for path in df.image_path:
        path = path.lstrip("/")
        rutas_nuevas.append(path)
    df.image_path = rutas_nuevas


def muestras_data_set(df,n, seed, use_seed = False):
    especies = pd.unique(df.label)
    df_final = pd.DataFrame(columns=df.columns)
    for especie in especies:
        df_fil = df[df.label == especie]
        if use_seed:
            df_fil = df_fil.sample(n=n, random_state=seed)
        else:
            df_fil = df_fil.sample(n=n)
        df_final = pd.concat([df_final, df_fil], ignore_index=True)
    return df_final

In [ ]:
rutas_val = pd.read_csv("kaggle/working/val.csv")
rutas_train = pd.read_csv("kaggle/working/train.csv")
rutas_test = pd.read_csv("kaggle/working/test.csv")
info_especies = pd.read_csv("InfoEspecies.csv")
info_especies = info_especies.drop(info_especies.columns[0], axis= "columns")

#rutas_train = muestras_data_set(df = rutas_train,n = 50, seed = 666, use_seed= True)
#rutas_test = muestras_data_set(df = rutas_test,n = 5, seed = 666, use_seed= True)
#rutas_val = muestras_data_set(df = rutas_val,n = 5, seed = 666, use_seed= True)

train = pd.merge(info_especies, rutas_train, on='label', how='inner')
test = pd.merge(info_especies, rutas_test, on='label', how='inner')
val = pd.merge(info_especies, rutas_val, on='label', how='inner')


dfs = [train, val, test]
for df in dfs:
    ArreglarLasRutas(df)

In [ ]:
def Modificar_direc(df_rutas, subset):
    os.makedirs(f"Data/{subset}", exist_ok=True)

    ArreglarLasRutas(df_rutas)

    for path, label in zip(df_rutas.image_path, df_rutas.label):
        destino = f"Data/{subset}/{label}/"
        os.makedirs(destino, exist_ok=True)
        print(f"Copiando {path} --> {destino}")
        shutil.copy(path, destino)
        print(f"Terminado {label}")

In [ ]:
Modificar_direc(test, "test")

In [ ]:
IMG_SIZE = (640, 480)
BATCH_SIZE = 64

test_dataset = tf.keras.utils.image_dataset_from_directory(
    "Data/test",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=False
)

In [ ]:
model = keras.models.load_model("EffTune.keras")

In [ ]:
from sklearn.metrics import confusion_matrix

y_true = np.concatenate([y for x, y in test_dataset], axis=0)
y_true_classes = np.argmax(y_true, axis=1)

y_pred = model.predict(test_dataset)
cm = confusion_matrix(y_true_classes, np.argmax(y_pred, axis=1),
                      normalize= "true")


In [ ]:
plt.figure(figsize=(64,64))
sns.heatmap(cm, annot= True)
plt.show()

In [ ]:
model.evaluate(test_dataset)